# RNA 3D Pilot10 — Independent Metric Validation (CPU)

**Purpose:** check whether the low TM-scores are compatible with the underlying structures or reflect a comparison problem. The notebook **does not alter original predictions or archived evaluations**. It reads the same 10 experimental native targets and matched predicted structures for **NuFold, DRfold2, and RhoFold+**, validates their coordinates, and performs an independent metric audit.

**Outputs:** 30-structure CSV with **C3′/C4′ lDDT-style single-atom-per-nucleotide diagnostics**, local/nonlocal contact counts, **independent Kabsch C3′ RMSD**, fixed-distance-scale C3′ similarity diagnostics, 3D overlays for three representative targets, and an immutable provenance JSON. Run in **Google Colab → Runtime → Change runtime type → None (CPU)**.

> Important: These *single-atom* lDDT-style checks are **not** the official RNA-Puzzles all-atom lDDT; the **fixed-d0** score is a diagnostic after RMSD-optimal superposition, **not** a replacement for US-align TM-score. **Official interaction-network fidelity (INF) is not calculated** here because RNA-Puzzles requires a validated base-pair/stacking annotation pipeline (e.g., MC-Annotate) and proper interaction-type comparison. We must not substitute an arbitrary distance/contact F1 and call it INF.

The run is exploratory and remains separate from full-MSA/template-scarcity conclusions.

**References**
- RNA-Puzzles assessment: https://github.com/RNA-Puzzles/RNA_assessment
- US-align method: https://pmc.ncbi.nlm.nih.gov/articles/PMC13373882/
- RNA-Puzzles Round V (INF and metrics): https://pmc.ncbi.nlm.nih.gov/articles/PMC11810798/

## 1. Mount Google Drive (no GPU)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted. CPU is sufficient.')

## 2. Frozen inputs, output-only new run ID, and imports

In [ ]:
from pathlib import Path
import csv, json, math, hashlib, shutil
from datetime import datetime, timezone
from statistics import mean, median
import numpy as np
import matplotlib.pyplot as plt

ROOT=Path('/content/drive/MyDrive/RNA3D_Benchmark')
NU=ROOT/'NuFold/runs/pilot10_single_20261018T203832Z'
DR=ROOT/'DRfold2/runs/drfold2_pilot10_20261110_v1'
RH=ROOT/'RhoFold/runs/rhofoldplus_pilot10_single_relax1000_20261111_v1'
NATIVE=ROOT/'native/pilot_v1/native_eval'
THREEWAY=RH/'evaluations/rhofoldplus_threeway_usalign20260527_20261112_v1'
PREV_DR=DR/'evaluations/drfold2_vs_nufold_usalign20260527_20261111_v1'
TARGETS=['8ubd_H','9fia_bJ','9cfn_B','9m31_B','9g4p_B',
         '9lq7_K','8z8u_B','9g6k_lB','9cf2_W','8wf8_B']
MODELS=['NuFold','DRfold2','RhoFold+ (single)']

RUN_ID='pilot10_independent_metric_diagnostics_20261113_v1'
OUTPUT_DIR=ROOT/'Evaluations'/RUN_ID
STAGE=Path('/content/rna3d_pilot10_metric_validation_stage_20261113')
assert len(TARGETS)==len(set(TARGETS))==10
for d in [NU,DR,RH,NATIVE,THREEWAY,PREV_DR]:
    assert d.is_dir(),f'Missing directory: {d}'
assert not OUTPUT_DIR.exists(),f'Run already exists; do not overwrite: {OUTPUT_DIR}'
assert not STAGE.exists(),f'Staging exists; inspect rather than overwrite: {STAGE}'
print('Existing evaluation:',THREEWAY)
print('NEW output (will be created only at final cell):',OUTPUT_DIR)
print('GPU required: NO; model inference: NO')

## 3. Verify immutable previous scores, model provenance, and target coverage

In [ ]:
def sha256_file(p):
    h=hashlib.sha256()
    with Path(p).open('rb') as f:
        for block in iter(lambda:f.read(1024*1024),b''): h.update(block)
    return h.hexdigest()

def read_csv(p):
    with Path(p).open(newline='') as f:return list(csv.DictReader(f))

archive=json.loads((THREEWAY/'summary.json').read_text())
finalized=json.loads((THREEWAY/'FINALIZED.json').read_text())
assert finalized['status']=='complete' and archive['counts']=={'targets_scored':10,'nufold_calibration_pass':10,'drfold2_calibration_pass':10}
for name,sha in finalized['files'].items():
    assert sha256_file(THREEWAY/name)==sha, f'Archived file mutated: {name}'
paired=read_csv(THREEWAY/'three_model_paired.csv')
assert len(paired)==10 and [x['target_id'] for x in paired]==TARGETS
PAIRED={x['target_id']:x for x in paired}
for t in TARGETS:
    assert (NATIVE/f'{t}_residue_map.csv').is_file()
    assert (NATIVE/f'{t}_native_eval.pdb').is_file()
    assert (NU/f'evaluations/pilot10_native_eval_v2/{t}/{t}_rank_1_mapped.pdb').is_file()
    assert (PREV_DR/f'mapped/{t}_mapped.pdb').is_file()
    assert (THREEWAY/f'mapped/{t}_mapped.pdb').is_file()
print('ARCHIVE HASH VERIFICATION PASS (5/5)')
print('THREE-MODEL INPUT CHECK PASS: 10 targets × 3 existing mapped structures')
print('Archive evaluation ID:',archive['evaluation_id'])
print('Scored native residues:',[int(PAIRED[t]['native_residues']) for t in TARGETS])

## 4. Independent PDB parser and atom correspondence

RNA native coordinates and model PDBs use matching **target residue numbers**. Check them strictly before analyzing distances. If any mismatch appears, stop rather than trying to guess an alignment.

In [ ]:
def pdb_atoms(path):
    out={};resnames={}
    for line in Path(path).read_text(errors='replace').splitlines():
        if not line.startswith(('ATOM  ','HETATM')):continue
        if len(line)<54:raise ValueError(f'Truncated PDB atom line: {path}')
        if line[16:17] not in (' ','A','1'):continue
        atom=line[12:16].strip();base=line[17:20].strip().upper()
        chain=line[21:22];rn=int(line[22:26]);icode=line[26:27]
        assert not icode.strip(),f'Unexpected insertion code: {path} {rn}'
        xyz=np.array([float(line[i:i+8]) for i in (30,38,46)],dtype=float)
        assert np.all(np.isfinite(xyz)),f'Nonfinite coordinates: {path}'
        if rn not in out:out[rn]={};resnames[rn]=(chain,base)
        assert resnames[rn]==(chain,base),f'Duplicate/conflicting chain/residue ID: {path}:{rn}'
        if atom not in out[rn]:out[rn][atom]=xyz
    assert out,f'Empty coordinates: {path}'
    assert len(set(x[0] for x in resnames.values()))==1,f'Multiple chains in {path}'
    return out,resnames

def expected_native_positions(t):
    rows=read_csv(NATIVE/f'{t}_residue_map.csv')
    keep=[int(r['target_position']) for r in rows if r['coordinates_present'].strip().lower()=='yes']
    assert keep==sorted(set(keep)),f'{t}: invalid native mapping'
    return keep

def paths(t):
    return {'Native':NATIVE/f'{t}_native_eval.pdb',
        'NuFold':NU/f'evaluations/pilot10_native_eval_v2/{t}/{t}_rank_1_mapped.pdb',
        'DRfold2':PREV_DR/f'mapped/{t}_mapped.pdb',
        'RhoFold+ (single)':THREEWAY/f'mapped/{t}_mapped.pdb'}

BASE_CODES={'A':'A','C':'C','G':'G','U':'U','RA':'A','RC':'C','RG':'G','RU':'U',
            'ADE':'A','CYT':'C','GUA':'G','URA':'U'}
def canonical_base(name):
    assert name in BASE_CODES,f'Unknown nucleotide residue code: {name}'
    return BASE_CODES[name]

def coords_for(t,model,atom):
    fs=paths(t)
    ref,refname=pdb_atoms(fs['Native']); pred,predname=pdb_atoms(fs[model])
    ids=expected_native_positions(t)
    assert set(ref)==set(pred)==set(ids),(t,model,'residue numbering mismatch')
    assert int(PAIRED[t]['native_residues'])==len(ids)
    for p in ids:
        assert canonical_base(refname[p][1])==canonical_base(predname[p][1]),(t,model,p,'base identity mismatch')
        assert atom in ref[p] and atom in pred[p],(t,model,p,f'missing {atom}')
    a=np.vstack([ref[p][atom] for p in ids])
    b=np.vstack([pred[p][atom] for p in ids])
    return np.asarray(ids),a,b

for t in TARGETS:
    for model in MODELS:
        for atom in ("C3'","C4'"):
            ids,ref,pred=coords_for(t,model,atom)
            assert len(ids)==int(PAIRED[t]['native_residues'])
print('PDB MAPPING/SEQUENCE/ATOM CHECK PASS: 10 × 3 × 2')

## 5. Metrics and synthetic correctness tests

**Kabsch RMSD** independently computes a least-squares superposition of the **corresponding C3′ atoms**; it may differ slightly from a US-align TM-optimized superposition. **lDDT-style** uses native pairs within 15 Å (standard tolerances 0.5/1/2/4 Å) and reports separate stricter nonlocal contacts (`|target_position_i - target_position_j| > 3`). Pairs are drawn from the experimental reference, so a missing native segment cannot be assessed. **Fixed-d0** is a diagnostic computed under the same Kabsch rotation; it is *not a re-scored US-align result*.

In [ ]:
def kabsch_superpose(pred,ref):
    assert pred.shape==ref.shape and pred.ndim==2 and pred.shape[1]==3
    pc,rc=pred.mean(0),ref.mean(0)
    uu,ss,vh=np.linalg.svd((pred-pc).T@(ref-rc))
    corr=np.eye(3);corr[-1,-1]=np.sign(np.linalg.det(uu@vh))
    rot=uu@corr@vh
    new=(pred-pc)@rot+rc
    rmsd=np.sqrt(np.mean(np.sum((new-ref)**2,axis=1)))
    return new,float(rmsd)

def local_distance_diagnostic(pos,ref,pred,seqsep=1,cutoff=15.0,thresholds=(0.5,1.0,2.0,4.0)):
    i,j=np.triu_indices(len(pos),1)
    dref=np.linalg.norm(ref[i]-ref[j],axis=1)
    sel=(dref<cutoff)&(np.abs(pos[j]-pos[i])>seqsep)
    i,j=i[sel],j[sel];dref=dref[sel]
    if len(i)==0:return None,0
    dmodel=np.linalg.norm(pred[i]-pred[j],axis=1)
    diff=np.abs(dmodel-dref)
    score=float(np.mean([np.mean(diff<t) for t in thresholds]))
    assert 0<=score<=1
    return score,len(i)

def fixed_d0_similarity(ref,pred_superposed,d0=2.0):
    d=np.linalg.norm(ref-pred_superposed,axis=1)
    return float(np.mean(1/(1+(d/d0)**2)))

# Unit tests: rotation and translation do not affect RMSD/lDDT.
rng=np.random.default_rng(20261113)
test=rng.normal(size=(18,3))*3
rot=np.linalg.qr(rng.normal(size=(3,3)))[0]
if np.linalg.det(rot)<0:rot[:,0]*=-1
shifted=test@rot+np.array([13.,-20.,4.])
xyz,rms=kabsch_superpose(shifted,test)
assert rms<1e-10
q0,cnt=local_distance_diagnostic(np.arange(1,19),test,shifted)
assert cnt>0 and abs(q0-1)<1e-12
assert abs(fixed_d0_similarity(test,xyz)-1)<1e-12
mut=test.copy();mut[5]+=np.array([8.,-5.,2.])
q1,cnt1=local_distance_diagnostic(np.arange(1,19),test,mut)
assert q1 < q0
print('SYNTHETIC METRIC TESTS: PASS; clean structure=1, perturbed structure<1, rigid motions invariant')

# Two archived published log checks: independent C3’ RMSDs (previous verified).
for target,expected in [('8ubd_H',1.93),('9m31_B',7.00)]:
    _,r,p=coords_for(target,'RhoFold+ (single)',"C3'")
    _,rms=kabsch_superpose(p,r)
    assert abs(rms-expected)<.03,(target,rms,expected)
    print(target,'independent C3 RMSD',round(rms,4),'Å; archive US-align',expected,'Å')

## 6. Score all 30 structures **without rerunning inference**

In [ ]:
records=[]
for t in TARGETS:
    native_n=int(PAIRED[t]['native_residues'])
    observed_frac=float(PAIRED[t]['native_observed_fraction'])
    for model in MODELS:
        pos,nat,pred=coords_for(t,model,"C3'")
        fitted,rmsd=kabsch_superpose(pred,nat)
        ld,contacts=local_distance_diagnostic(pos,nat,pred,seqsep=1)
        ld_far,contacts_far=local_distance_diagnostic(pos,nat,pred,seqsep=3)
        pos4,nat4,pred4=coords_for(t,model,"C4'")
        assert (pos4==pos).all()
        ld4,contacts4=local_distance_diagnostic(pos4,nat4,pred4,seqsep=1)
        archive_col={'NuFold':'nufold_rank1','DRfold2':'drfold2','RhoFold+ (single)':'rhofoldplus_single'}[model]
        row={
            'target_id':t,'model':model,'native_residues':native_n,
            'native_observed_fraction':observed_frac,
            'archived_tm_score':float(PAIRED[t][archive_col+'_tm_score']),
            'archived_usalign_rmsd_angstrom':float(PAIRED[t][archive_col+'_rmsd_angstrom']),
            'independent_kabsch_c3_rmsd_angstrom':rmsd,
            'c3_lddt_style_seqsep_gt1':ld,
            'c3_lddt_style_local_pair_count':contacts,
            'c3_lddt_style_seqsep_gt3':ld_far,
            'c3_lddt_style_nonlocal_pair_count':contacts_far,
            'c4_lddt_style_seqsep_gt1':ld4,
            'c4_lddt_style_pair_count':contacts4,
            'c3_fixed_d0_0p4_kabsch_diagnostic':fixed_d0_similarity(nat,fitted,d0=.4),
            'c3_fixed_d0_2p0_kabsch_diagnostic':fixed_d0_similarity(nat,fitted,d0=2.0),
            'native_pdb_sha256':sha256_file(paths(t)['Native']),
            'mapped_model_pdb_sha256':sha256_file(paths(t)[model])}
        assert rmsd<=row['archived_usalign_rmsd_angstrom']+.06, (t,model,'independent RMSD unexpectedly exceeds US-align',rmsd,row['archived_usalign_rmsd_angstrom'])
        records.append(row)
assert len(records)==30
for t in TARGETS:
    assert len([r for r in records if r['target_id']==t])==3
print('INDEPENDENT METRIC AUDIT PASS:',len(records),'model-target combinations')
for target in ['8ubd_H','9m31_B','9cfn_B']:
    print('\n',target)
    for r in records:
        if r['target_id']==target:
            print(f" {r['model']:18} TM={r['archived_tm_score']:.5f} RMSD(C3)={r['independent_kabsch_c3_rmsd_angstrom']:.2f} C3-lDDT-style={r['c3_lddt_style_seqsep_gt1']:.3f} local pairs={r['c3_lddt_style_local_pair_count']} nonlocal pairs={r['c3_lddt_style_nonlocal_pair_count']}")

## 7. Completeness check and model aggregates

Do not average undefined nonlocal lDDT: if no eligible native reference pairs exist, report `not estimable`, not zero or perfect agreement.

In [ ]:
from collections import defaultdict

def avg(items,key):
    vals=[float(x[key]) for x in items if x[key] is not None]
    return mean(vals) if vals else None

summary={}
for subset,rs in [('all_ten_exploratory',records),
                  ('native_observed_fraction_ge_0p80',[r for r in records if r['native_observed_fraction']>=.8])]:
    summary[subset]={}
    for model in MODELS:
        m=[r for r in rs if r['model']==model]
        summary[subset][model]={
            'targets':len(m),
            'mean_archived_tm_score':avg(m,'archived_tm_score'),
            'mean_c3_lddt_style_seqsep_gt1':avg(m,'c3_lddt_style_seqsep_gt1'),
            'mean_c3_lddt_style_seqsep_gt3':avg(m,'c3_lddt_style_seqsep_gt3'),
            'nonlocal_evaluable_targets':sum(r['c3_lddt_style_seqsep_gt3'] is not None for r in m),
            'mean_independent_c3_rmsd_angstrom':avg(m,'independent_kabsch_c3_rmsd_angstrom'),
            'mean_c3_fixed_d0_2p0_kabsch_diagnostic':avg(m,'c3_fixed_d0_2p0_kabsch_diagnostic')}
    print('\n',subset)
    for name,s in summary[subset].items():
        print(f"{name:20} n={s['targets']:2} TM={s['mean_archived_tm_score']:.4f} C3-lDDT-style={s['mean_c3_lddt_style_seqsep_gt1']:.4f} nonlocal={s['mean_c3_lddt_style_seqsep_gt3']} C3-RMSD={s['mean_independent_c3_rmsd_angstrom']:.2f}")

# Strict test: we are using exactly the pre-existing frozen pairs.
for mod,key in [('NuFold','nufold_rank1'),('DRfold2','drfold2'),('RhoFold+ (single)','rhofoldplus_single')]:
    expected=float(archive['summary']['all_ten_exploratory'][key]['mean_tm_score'])
    assert abs(summary['all_ten_exploratory'][mod]['mean_archived_tm_score']-expected)<1e-10
print('ARCHIVED TM SCORE MEANS REPRODUCED: ALL 3 MODELS')

## 8. Structural overlays (three representative targets)

The overlays use independent C3′ least-squares superposition to the experimental structure, **not** the US-align rotation. Missing native residues are not drawn. Line segments break across gaps in experimental native coordinates. These views are qualitative aids, not evidence of formal INF or lDDT.

In [ ]:
def plot_overlay(t,out_path):
    fig=plt.figure(figsize=(9,7))
    ax=fig.add_subplot(111,projection='3d')
    model_paths=paths(t)
    pos,ref,_=coords_for(t,MODELS[0],"C3'")
    sets=[('Native',ref)]
    for model in MODELS:
        _,native,pred=coords_for(t,model,"C3'")
        aligned,_=kabsch_superpose(pred,native)
        sets.append((model,aligned))
    # Add break points when target sequence positions have missing experimental coordinates.
    coords=[]
    for label,xyz in sets:
        z=xyz.copy()
        # 3D line per consecutive block to avoid false covalent bridges across unresolved residues.
        cut=np.where(np.diff(pos)>1)[0]+1
        for chunk_idx,chunk in enumerate(np.split(z,cut)):
            ax.plot(chunk[:,0],chunk[:,1],chunk[:,2],lw=2.5 if label=='Native' else 1.45,
                    label=label if chunk_idx==0 else None)
        ax.scatter(z[:,0],z[:,1],z[:,2],s=10,alpha=.65)
        coords.append(z)
    allc=np.vstack(coords)
    low=allc.min(0); high=allc.max(0)
    center=(low+high)/2;size=float(max(high-low))
    ax.set(xlim=(center[0]-size*.55,center[0]+size*.55),
           ylim=(center[1]-size*.55,center[1]+size*.55),
           zlim=(center[2]-size*.55,center[2]+size*.55),
           xlabel='X (Å)',ylabel='Y (Å)',zlabel='Z (Å)')
    ax.set_box_aspect((1,1,1))
    ax.set_title(f'{t}: C3′ native vs 3 models, Kabsch fit; observed residues={len(pos)}')
    ax.legend(loc='upper left')
    plt.tight_layout()
    fig.savefig(out_path,dpi=145,bbox_inches='tight')
    plt.show()
    plt.close(fig)

STAGE.mkdir(parents=True)
(STAGE/'plots').mkdir()
for t in ['8ubd_H','9m31_B','9cfn_B']:
    plot_overlay(t,STAGE/'plots'/f'{t}_C3_overlay.png')
print('Saved 3 qualitative overlays in local staging.')

## 9. Save immutable validation outputs to a NEW Drive directory

**No original files are rewritten.** This cell writes a CSV, JSON provenance, three plots, and a checksummed completion marker under `MyDrive/RNA3D_Benchmark/Evaluations/<RUN_ID>`. If the destination already exists, stop and review it before choosing another run ID.

In [ ]:
def write_csv(path,rows):
    with path.open('w',newline='') as f:
        writer=csv.DictWriter(f,fieldnames=list(rows[0]),lineterminator='\n')
        writer.writeheader();writer.writerows(rows)

write_csv(STAGE/'pilot10_independent_metrics.csv',records)
metadata={
    'status':'complete','run_id':RUN_ID,
    'created_utc':datetime.now(timezone.utc).isoformat(),
    'source_threeway_evaluation':archive['evaluation_id'],
    'source_threeway_evaluation_hash':sha256_file(THREEWAY/'summary.json'),
    'source_git_mapping_commit':archive['pinned_mapping_project_commit'],
    'target_ids':TARGETS,'model_names':MODELS,
    'archive_usalign_version':archive['usalign_tag'],
    'methods':{
        'reference_policy':'Native observed target positions, exact target residue numbers and nucleotide identities',
        'kabsch_rmsd':'C3-prime atoms, identity target index correspondence, proper 3D rotation and translation',
        'lddt_style':'Single C3-prime or C4-prime per residue; native distances <15Å, |sequence position delta|>1; tolerances 0.5,1,2,4Å. Nonlocal diagnostic excludes |delta|<=3.',
        'fixed_d0':'C3-prime similarity computed from Kabsch fitted coordinates (not US-align TMscore); scale d0=0.4Å and 2.0Å, normalized by number of native-observed residues',
        'INF':'NOT CALCULATED; external validated RNA-Puzzles base-pair/noncanonical/stacking interaction extraction and comparator required.'},
    'summary':summary,
    'research_caveats':[
        'Exploratory sample of 10; do not use to claim model rankings.',
        'This single-atom lDDT-style measure is NOT official all-atom lDDT or published RNA-adapted lDDT_RNA.',
        'Small number of native local pairs can make local score unstable; always show pair counts.',
        'No INF, canonical WC, non-WC, stacking INF, or all-atom lDDT reported.',
        'Short target and incomplete experimental coordinate bias remain; 8ubd_H has only 13/36 native-resolved residues.',
        'RhoFold+ is sequence-only quick-reference, not full MSA. Model inputs differ.',
        'This is an evaluation audit; it does not rerun model inference or independently validate experimental chain provenance / training leakage.']}
(STAGE/'summary.json').write_text(json.dumps(metadata,indent=2)+'\n')
required=['pilot10_independent_metrics.csv','summary.json']+[f'plots/{t}_C3_overlay.png' for t in ['8ubd_H','9m31_B','9cfn_B']]
assert len(read_csv(STAGE/required[0]))==30
assert not OUTPUT_DIR.exists(),f'Output exists; refuse overwrite: {OUTPUT_DIR}'
OUTPUT_DIR.parent.mkdir(parents=True,exist_ok=True)
shutil.copytree(STAGE,OUTPUT_DIR)
for rel in required:
    assert sha256_file(STAGE/rel)==sha256_file(OUTPUT_DIR/rel),f'Copy checksum failure: {rel}'
(OUTPUT_DIR/'FINALIZED.json').write_text(json.dumps({
    'status':'complete','run_id':RUN_ID,
    'files':{rel:sha256_file(OUTPUT_DIR/rel) for rel in required},
    'created_utc':datetime.now(timezone.utc).isoformat()},indent=2)+'\n')
print('INDEPENDENT VALIDATION COMPLETE: 30/30')
print('SAVED, VERIFIED:',OUTPUT_DIR)
print('NO MODEL INFERENCE; NO GITHUB WRITE')

## What we can conclude after running

1. **Scoring pipeline audit:** Comparing the independently computed C3′ Kabsch RMSDs against archived US-align RMSDs provides a useful error check. They can differ slightly because the optimization objectives need not be identical.
2. **Short-RNA sensitivity:** The native-length-aware US-align TM-score penalizes deviations at a very small RNA-specific `d0` for short references. High local agreement with low TM-score is possible and should be interpreted with native-coverage and local-pair counts.
3. **Generalization:** A target's high C3′-style lDDT does **not** prove correct base pairing, noncanonical contacts, or full RNA tertiary topology. The absence of nonlocal native contacts can make nonlocal lDDT **not estimable**.
4. **INF remains pending:** Use the RNA-Puzzles assessment pipeline with MC-Annotate and validated nucleotide numbering before reporting `INF_WC`, `INF_nWC`, or `INF_stack`. For now, **do not fabricate INF**. Official RNA-Puzzles workflow: https://github.com/RNA-Puzzles/RNA_assessment
5. **Next design step:** Extend the dataset to longer, more complete RNA native structures, and screen model training/test overlap and template scarcity separately.

**Do not push these exploratory validation results to GitHub until explicitly approved.**